# Fine-tuning LFM2.5-350M for schema-compliant structured output with GRPO

### The official recipe, with an unimpeachable evaluator bolted on

This notebook merges two implementations of the recipe in
[*Fine-tuning a 350M Model for Better Structured Outputs in 100 GRPO Steps*](https://huggingface.co/blog/grpo-with-trl-ifstruct)
(Monigatti, Burtenshaw & Paniego — Hugging Face, 3 September 2026):

* **The training half is the official one.** Data source, filters, augmentation, LoRA
  targets, all three reward functions and every hyperparameter are taken verbatim from
  Liquid's [`cookbook/finetuning/notebooks/grpo_with_trl_ifstruct.ipynb`](https://github.com/Liquid4All/cookbook/blob/main/finetuning/notebooks/grpo_with_trl_ifstruct.ipynb).
* **The evaluation half calls Liquid's own scorer.** Instead of re-implementing the
  IFStruct checks — which is easy to get subtly wrong — we `pip install` the
  [`ifstruct`](https://github.com/Liquid4All/ifstruct) package and call
  `validate_response()`, the function behind the published numbers.

**Reported result:** IFStruct **22.6% (452/2000) → 29.7% (594/2000)** after ~500 rows and
100 GRPO steps. JSON drove it (**18.0% → 31.9%**); YAML was flat (**27.2% → 27.5%**)
because the recipe trains JSON only. A later, larger Liquid RL run reached **44.9%** —
a different procedure, not this notebook.

> Liquid's own notebook states plainly: *"this is not the training pipeline used to train
> the RL model described in the IFStruct blog post. The goal of this notebook is not to
> recreate the IFStruct benchmark score but to show how fine-tuning can improve eval
> scores."* Treat 29.7% as a reference point, not a target you must hit.

---

### Getting the facts right

These five points are wrong in most circulating summaries of this recipe. Each was
checked against primary sources before this notebook was written.

| Item | The actual recipe |
|---|---|
| **Training data** | `nvidia/Nemotron-RL-instruction_following-structured_outputs`, first 1,000 rows filtered to ~500. **Not** IFStruct, and not synthetic. |
| **Held-out data** | `LiquidAI/ifstruct-v1.0` — 2,000 frozen **test** prompts. Never train on it. |
| **Precision** | FP16 on T4 / BF16 where supported. **No 4-bit quantization** — 350M in fp16 is ~0.7 GB. |
| **LoRA targets** | `q_proj, k_proj, v_proj, out_proj, in_proj, w1, w2, w3`. LFM2's attention output projection is **`out_proj`, not `o_proj`** — PEFT silently ignores names that match nothing. |
| **Schema shape** | IFStruct's `json_schema` root is **always `{"type": "array"}`**, describing the *unwrapped* list. Validating it per-item fails every row. |

### Scope

This trains **form, not substance**. IFStruct scores only whether output parses and
matches the requested shape — never whether the content is correct. LFM2.5-350M is built
for extraction, structured output and tool use; it is the wrong base for knowledge-heavy
generation, and no amount of format reward changes that.

> **Runtime → Change runtime type → T4 GPU** before running anything.

## 1. Why GRPO works here

SFT maximises the likelihood of reference tokens. It can teach a model what schema-shaped
text *looks like*, but it cannot express *"this must parse"* or *"this field must be an
integer in [1, 8]"* — those are properties of the whole decoded string, checkable only
after generation.

GRPO lives exactly there. For each prompt it samples $G$ completions, scores them with a
program, and makes the group its own baseline:

$$\hat A_i = \frac{r_i - \operatorname{mean}(r)}{\operatorname{std}(r) + \epsilon}$$

Above-average completions get pushed up, below-average ones down. **No critic network is
trained**, which is why this fits on a free T4.

The corollary matters more than the formula: **if every completion in a group scores the
same, every advantage is zero and the step teaches nothing.** That single fact explains
two design choices you will see below — hot sampling (`temperature=1.1`) to keep groups
varied, and *partial-credit* rewards rather than one binary pass/fail.

### The failure modes being trained away

| Failure | What it looks like |
|---|---|
| Unparseable | truncated JSON, trailing prose, backtick spam |
| Wrong container | bare `[...]` when `{"key": [...]}` was demanded, or vice versa |
| Fencing | missing ```` ```json ```` fence, or a fence when raw output was demanded |
| Commentary | *"Sure! Here's your JSON:"* |
| Schema drift | invented keys, `"3"` where an integer was required, enum near-misses |
| Miscount | 4 items when the prompt said 2 |

## 2. Environment

Versions are pinned to the official notebook's. Pinning matters: TRL's GRPO API has moved
fast, and an unpinned install can silently invalidate the recipe.

We add one dependency the official notebook does not have — `ifstruct` itself, straight
from Liquid's repo. It is pure Python (PyYAML + requests) and ships both the validator
and the frozen 2,000-row test set.

In [ ]:
%pip install -q "trl==1.7.1" "transformers==5.13.0" "peft==0.19.1" "torchao>=0.16.0" \
                datasets accelerate jsonschema pyyaml matplotlib pandas
# Liquid's official evaluator + frozen test set -- this is our metric, not a re-implementation
%pip install -q "git+https://github.com/Liquid4All/ifstruct.git"

In [ ]:
import json, math, os, random, re, time
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
import torch
import transformers
from datasets import load_dataset
from jsonschema import Draft7Validator

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
transformers.set_seed(SEED)
random.seed(SEED)

assert torch.cuda.is_available(), "GPU required: Runtime -> Change runtime type -> T4 GPU"

CAPABILITY = torch.cuda.get_device_capability(0)
# bf16 needs Ampere (SM 8.0+). A T4 is Turing (7.5) -> fp16 only. Forcing bf16 there is
# the most common reason this recipe produces NaNs or crawls.
USE_BF16 = CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print({"gpu": torch.cuda.get_device_name(0),
       "sm": f"{CAPABILITY[0]}.{CAPABILITY[1]}",
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
       "bf16": USE_BF16, "dtype": str(DTYPE).split(".")[-1],
       "torch": torch.__version__, "transformers": transformers.__version__})

### Run mode

Run `SMOKE_TEST=True` first — always. It exercises every cell in minutes and catches the
expensive class of bug (a reward that returns a constant, a column that got dropped, an
OOM) before you commit hours. It makes **no** benchmark claim.

`SMOKE_TEST=False` is the published experiment: 100 steps, 8 generations, ~500 rows.
The authors report roughly **4 hours on a T4**, ~1.5 hours on an A100.

In [ ]:
SMOKE_TEST = True    # FIRST RUN: keep True. Flip only after the pipeline completes.

if SMOKE_TEST:
    TRAIN_SOURCE_ROWS, MAX_STEPS, NUM_GENERATIONS = 120, 5, 2
    MAX_COMPLETION_LENGTH, N_EVAL, RUN_NAME = 384, 24, "smoke"
else:
    TRAIN_SOURCE_ROWS, MAX_STEPS, NUM_GENERATIONS = 1000, 100, 8   # filters to ~500 rows
    MAX_COMPLETION_LENGTH, N_EVAL, RUN_NAME = 1024, 150, "full-100-step"

MODEL_ID   = "LiquidAI/LFM2.5-350M"
DATASET_ID = "nvidia/Nemotron-RL-instruction_following-structured_outputs"
OUTPUT_DIR = f"./outputs/lfm25-350m-structured-grpo-{RUN_NAME}"

# --- evaluation contract (identical before and after; see section 6) -----------------
EVAL_SEED = 1234      # fixes WHICH test rows are scored
EVAL_MAX_NEW_TOKENS = 768   # official harness uses 2048+; a T4 subset run cannot afford it
EVAL_BATCH_SIZE = 8

print({"mode": RUN_NAME, "source_rows": TRAIN_SOURCE_ROWS, "steps": MAX_STEPS,
       "generations": NUM_GENERATIONS, "max_completion": MAX_COMPLETION_LENGTH,
       "eval_rows": N_EVAL})

## 3. The metric, before anything else

IFStruct is 2,000 frozen test prompts. Each response passes through six checks and
**passes only if all six produce zero errors** — score 1 or 0, no partial credit.

| # | Check | Typical failure |
|---|-------|-----------------|
| 1 | **Parse** | truncation, trailing text, JSON inside a `yaml` fence |
| 2 | **Code block** | fence required but absent |
| 3 | **No commentary** | text before/after the document |
| 4 | **Structure** | bare list vs `{"key": [...]}` wrapper, wrong key name |
| 5 | **Item count** | exact `n`, or a `[min, max]` range |
| 6 | **Schema** | types, required fields, enums, bounds, **and extraneous keys** |

Two details are load-bearing, and both are easy to get wrong in a hand-rolled scorer:

* **`json_schema` is always `{"type": "array", ...}`** — the *unwrapped* list, not the
  wrapper object. Applying it per item fails every row.
* **A YAML request cannot be satisfied with JSON.** YAML is a superset of JSON, so JSON
  parses cleanly as YAML; the validator explicitly rejects flow-style mappings.

Rather than reproduce that logic, we call it.

In [ ]:
from ifstruct.validator import validate_response
import ifstruct

def load_ifstruct_test():
    """Frozen 2,000-row test set: from the installed package, else straight from GitHub."""
    local = Path(ifstruct.__file__).resolve().parent.parent / "data" / "test.jsonl"
    if local.exists():
        raw = local.read_text()
    else:
        import urllib.request
        raw = urllib.request.urlopen(
            "https://raw.githubusercontent.com/Liquid4All/ifstruct/main/data/test.jsonl"
        ).read().decode()
    return [json.loads(line) for line in raw.splitlines() if line.strip()]

TEST_ROWS = load_ifstruct_test()

def _decode(value):
    """IFStruct fields arrive as native objects from JSONL but as JSON strings from the
    Hub (Arrow cannot hold `int | list` or heterogeneous schemas in one column).
    Accept both -- assuming one shape is a real, silent source of breakage."""
    return json.loads(value) if isinstance(value, str) else value

def score_one(row, response):
    """The official IFStruct verdict for one response."""
    return validate_response(
        response=response,
        json_schema=_decode(row["json_schema"]),
        top_level_count=_decode(row["top_level_count"]),
        require_no_commentary=bool(row["require_no_commentary"]),
        output_format=row["output_format"],
        top_level_key=row["top_level_key"],
        require_wrapper_key=bool(row["require_wrapper_key"]),
        require_code_block=bool(row["require_code_block"]),
    )

print(f"IFStruct test rows: {len(TEST_ROWS)}")
for col in ["output_format", "require_wrapper_key", "require_code_block", "require_no_commentary"]:
    print(f"  {col:24s} {dict(Counter(r[col] for r in TEST_ROWS))}")
print("  schema root types        ",
      dict(Counter(_decode(r["json_schema"]).get("type") for r in TEST_ROWS)))

### Prove the harness works before trusting a number from it

`reference_answer()` fills a schema with dummy values and renders it in the requested
shape. It is not a model — it is a constructive proof that a row is satisfiable, and a
test of whether *we* understand the spec. If it does not pass 100% of the real test set,
our reading of IFStruct is wrong and every score downstream is meaningless.

In [ ]:
import yaml

def _instance(schema, rng, depth=0):
    if "enum" in schema:
        return rng.choice(schema["enum"])
    t = schema.get("type")
    if t == "array":
        lo = schema.get("minItems", 1); hi = schema.get("maxItems", lo)
        return [_instance(schema["items"], rng, depth + 1) for _ in range(rng.randint(lo, hi))]
    if t == "object":
        return {k: _instance(v, rng, depth + 1) for k, v in schema.get("properties", {}).items()}
    if t == "integer":
        return rng.randint(int(schema.get("minimum", 0)), int(schema.get("maximum", 100)))
    if t == "number":
        return round(rng.uniform(float(schema.get("minimum", 0)), float(schema.get("maximum", 100))), 2)
    if t == "boolean":
        return rng.choice([True, False])
    # quotes and newlines on purpose: escaping is one of IFStruct's hard axes
    return rng.choice(["Sample text", 'He said "ok" and left', "line one\nline two"])


def reference_answer(row, rng=None):
    rng = rng or random.Random(row.get("seed", 0))
    schema, count = _decode(row["json_schema"]), _decode(row["top_level_count"])
    items = _instance(schema, rng)
    n = count if isinstance(count, int) else rng.randint(count[0], count[1])
    while len(items) < n:
        items.append(_instance(schema["items"], rng))
    items = items[:n]

    payload = {row["top_level_key"]: items} if row["require_wrapper_key"] else items
    if row["output_format"] == "json":
        body, lang = json.dumps(payload, indent=2), "json"
    else:
        # block style only -- flow-style YAML is rejected as "JSON in disguise"
        body = yaml.safe_dump(payload, default_flow_style=False, sort_keys=False,
                              allow_unicode=True, width=10**6).rstrip()
        lang = "yaml"
    return f"```{lang}\n{body}\n```" if row["require_code_block"] else body


failures = [r for r in TEST_ROWS if not score_one(r, reference_answer(r)).passed]
print(f"constructed answers passing the official validator: "
      f"{len(TEST_ROWS) - len(failures)}/{len(TEST_ROWS)}")
assert not failures, "spec misunderstanding -- do not trust any score until this is 100%"

In [ ]:
# Mutate a perfect answer and watch the checks fall over one at a time.
demo = next(r for r in TEST_ROWS if r["output_format"] == "json" and r["require_wrapper_key"]
            and r["require_code_block"] and r["require_no_commentary"])
perfect = reference_answer(demo, random.Random(0))
items = json.loads(perfect.split("```json\n")[1].rsplit("\n```", 1)[0])[demo["top_level_key"]]
fence = lambda obj: "```json\n" + json.dumps(obj, indent=2) + "\n```"

mutations = {
    "perfect":                perfect,
    "chatty preamble":        "Sure! Here you go:\n\n" + perfect + "\n\nHope that helps!",
    "fence stripped":         perfect.replace("```json\n", "").replace("\n```", ""),
    "wrapper dropped":        fence(items),
    "wrong wrapper key":      fence({"data": items}),
    "item count doubled":     fence({demo["top_level_key"]: items * 2}),
    "extra key per item":     fence({demo["top_level_key"]: [dict(i, note="fyi") for i in items]}),
    "empty list":             fence({demo["top_level_key"]: []}),
    "truncated mid-object":   "```json\n" + json.dumps({demo["top_level_key"]: items})[:60],
    "schema echoed back":     fence(_decode(demo["json_schema"])),
}
print(f"{'mutation':24s} {'pass':>5s}   first error")
print("-" * 96)
for label, text in mutations.items():
    v = score_one(demo, text)
    print(f"{label:24s} {str(v.passed):>5s}   {(v.errors[0][:60] if v.errors else '')}")

print("\nNote 'extra key per item': the official validator flags extraneous fields even "
      "though\nIFStruct schemas do not set additionalProperties. A plain jsonschema call "
      "would pass it.")

## 4. Training data: Nemotron, cleaned and augmented

The training set is **not** IFStruct. It is NVIDIA's
`Nemotron-RL-instruction_following-structured_outputs`, which supplies a prompt, a JSON
Schema (`schema_str`), and the expected top-level field count (`schema_fields_count`).
IFStruct stays untouched as held-out test data.

**Data hygiene.** `LiquidAI/ifstruct-v1.0` is a frozen public *test* set. Do not train on
it, do not paraphrase its prompts, and do not tune reward thresholds against individual
test rows — that last one is the subtle version of the same mistake.

Two cleaning filters and one augmentation, exactly as in the official notebook:

* drop prompts over 6,000 characters (~1,600 tokens) to keep step time survivable;
* drop rows with `$ref` or otherwise invalid Draft 7 schemas;
* **augment deterministically by index** (no RNG, so it is reproducible): 40% get a
  fenced-output requirement, a disjoint 20% become bare-array tasks with an exact count,
  40% are left alone.

That augmentation is the whole reason the published gains concentrate in JSON fencing and
bare-list compliance: Nemotron prompts only ever ask for one raw JSON object, so without
it the model never practises either.

In [ ]:
train_ds = load_dataset(DATASET_ID, split=f"train[:{TRAIN_SOURCE_ROWS}]")
print(train_ds)
print("columns:", train_ds.column_names)

In [ ]:
def adapt_row(row: dict[str, Any]) -> dict[str, Any]:
    """GRPOTrainer wants a conversational `prompt` column; the rest reaches the rewards
    through **kwargs (GRPOConfig sets remove_unused_columns=False by default)."""
    return {"prompt": row["responses_create_params"]["input"],
            "wants_fence": False}          # ground truth flag; augment() flips some to True

train_ds = train_ds.map(adapt_row, remove_columns=["responses_create_params"])

def prompt_fits(row) -> bool:
    return sum(len(m["content"]) for m in row["prompt"]) <= 6000

def schema_is_valid(row) -> bool:
    if '"$ref"' in row["schema_str"]:
        return False
    try:
        Draft7Validator.check_schema(json.loads(row["schema_str"]))
        return True
    except Exception:
        return False

n0 = len(train_ds)
train_ds = train_ds.filter(prompt_fits);   n1 = len(train_ds)
train_ds = train_ds.filter(schema_is_valid); n2 = len(train_ds)
print({"loaded": n0, "after_length_filter": n1, "after_schema_filter": n2})
assert n2 > 0, "no usable rows survived filtering"

In [ ]:
def augment(row: dict[str, Any], idx: int) -> dict[str, Any]:
    """Deterministic, index-based. idx%5 in {0,1} -> fenced (40%);
    idx%5 == 2 -> top-level array of exactly n (20%); idx%5 in {3,4} -> unchanged (40%)."""
    variant = idx % 5
    if variant > 2:
        return row
    messages = [dict(m) for m in row["prompt"]]
    if variant == 2:
        n = 2 + idx % 3      # 2-4, varied so the count must be read from the prompt
        array_schema = {"type": "array", "items": json.loads(row["schema_str"]),
                        "minItems": n, "maxItems": n}
        messages[-1]["content"] += f"\n\nReturn a JSON array containing exactly {n} of these objects."
        return {**row, "prompt": messages, "schema_str": json.dumps(array_schema)}
    messages[-1]["content"] += "\n\nReturn the output inside a fenced code block."
    return {**row, "prompt": messages, "wants_fence": True}

train_ds = train_ds.map(augment, with_indices=True)

top_type = lambda s: json.loads(s).get("type")
display(pd.Series({
    "rows":   len(train_ds),
    "fenced": sum(train_ds["wants_fence"]),
    "array":  sum(top_type(s) == "array" for s in train_ds["schema_str"]),
    "object": sum(top_type(s) == "object" for s in train_ds["schema_str"]),
}).to_frame("count"))

In [ ]:
# Read one row before spending GPU time on it.
i = min(2, len(train_ds) - 1)
print("PROMPT:\n", train_ds[i]["prompt"][-1]["content"][:2000])
print("\nSCHEMA:\n", json.dumps(json.loads(train_ds[i]["schema_str"]), indent=2)[:1200])
print("\nwants_fence:", train_ds[i]["wants_fence"],
      "| expected fields/object:", train_ds[i]["schema_fields_count"])

## 5. Model and LoRA

The official run trains FP16/BF16 weights through a LoRA adapter — **no 4-bit
quantization**. At 350M parameters that is the right call: the model is ~0.7 GB in fp16,
so quantising buys memory you were never short of while slowing generation, which is
GRPO's actual bottleneck.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=DTYPE, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("dtype:", DTYPE, "| params:",
      f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M",
      "| allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 3))

### Confirm the LoRA target names against the checkpoint

LFM2 is a hybrid stack (short-convolution and GQA layers, assigned per layer by
`config.layer_types`). Its attention output projection is **`out_proj`** — not the
`o_proj` most LoRA snippets use. PEFT silently ignores target names that match nothing,
so a copied list can leave whole blocks untrained and your reward curve mysteriously
flat. Enumerate, don't trust.

In [ ]:
import torch.nn as nn

leaf_linears = Counter(name.split(".")[-1] for name, m in model.named_modules()
                       if isinstance(m, nn.Linear))
print("linear submodules in this checkpoint:", dict(leaf_linears))

TARGET_MODULES = ["q_proj", "k_proj", "v_proj",   # GQA projections
                  "out_proj", "in_proj",          # attention output + short-conv
                  "w1", "w2", "w3"]               # feed-forward
missing = [t for t in TARGET_MODULES if t not in leaf_linears]
assert not missing, f"these targets match nothing and would be silently ignored: {missing}"
print("o_proj present?", "o_proj" in leaf_linears, "| out_proj present?", "out_proj" in leaf_linears)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, lora_alpha=32, bias="none", task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()      # expect ~6M, ~1.7%

> **LoRA's `lora_B` is initialised to zeros**, so a freshly attached adapter is a
> mathematical no-op — the model right now *is* the base model. That is why the baseline
> in section 6 can be measured through the wrapped model, and why afterwards we can A/B
> the two with `model.disable_adapter()` instead of loading a second copy.

## 6. Baseline on real IFStruct — the evaluation contract

Three rules make a before/after comparison mean something. We fix them here and never
touch them again:

1. **The same rows both times** (`EVAL_SEED` fixes the sample).
2. **The same decoding both times** — greedy, same token cap.
3. **The official prompt shape**: a *single user message, no system prompt*. Liquid's
   harness sends the bare user turn at `temperature=0.0`. Adding a helpful
   *"return only valid JSON"* system prompt lifts the score for free and quietly destroys
   comparability with the published baseline.

Our absolute number still will not equal 22.6%: the official run scores all 2,000 rows
through llama.cpp with a much larger token budget (see section 11). **The delta is the
result.**

In [ ]:
@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE):
    """Greedy, left-padded batched generation; one user message, matching the harness."""
    model.eval()
    prev_pad, prev_trunc = tokenizer.padding_side, tokenizer.truncation_side
    tokenizer.padding_side = "left"        # right padding corrupts generation
    tokenizer.truncation_side = "left"     # never truncate away the generation prompt
    out = []
    try:
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                                   tokenize=False, add_generation_prompt=True)
                     for p in chunk]
            # add_special_tokens=False: the chat template already emitted BOS
            enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                            max_length=2048, add_special_tokens=False).to(model.device)
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, use_cache=True)
            out.extend(tokenizer.batch_decode(gen[:, enc["input_ids"].shape[-1]:],
                                              skip_special_tokens=True))
            print(f"  {min(i + batch_size, len(prompts))}/{len(prompts)}", end="\r")
    finally:
        tokenizer.padding_side, tokenizer.truncation_side = prev_pad, prev_trunc
    print()
    return out


def evaluate(rows, label, **kw):
    t0 = time.time()
    completions = generate_batch([r["prompt"] for r in rows], **kw)
    records = []
    for r, c in zip(rows, completions):
        v = score_one(r, c)
        records.append({"row": r, "completion": c, "passed": v.passed, "errors": v.errors,
                        "ratio": v.details.get("schema_match_ratio", 0.0)})
    rate = sum(x["passed"] for x in records) / len(records)
    print(f"=== {label}: {sum(x['passed'] for x in records)}/{len(records)} "
          f"passed ({100 * rate:.1f}%)  [{time.time() - t0:.0f}s]")
    return rate, records


def report(records, label):
    print(f"\n--- {label} ---")
    for axis, fn in [("format",    lambda r: r["row"]["output_format"]),
                     ("structure", lambda r: "wrapper" if r["row"]["require_wrapper_key"] else "bare list"),
                     ("fenced",    lambda r: f"fence={r['row']['require_code_block']}")]:
        g = {}
        for rec in records:
            g.setdefault(fn(rec), []).append(rec["passed"])
        print(f"  by {axis:10s} " + "  ".join(
            f"{k}: {100 * sum(v) / len(v):5.1f}% (n={len(v)})" for k, v in sorted(g.items())))
    errs = Counter()
    for rec in records:
        for e in rec["errors"]:
            errs[re.sub(r"['\"].*", "", e)[:56]] += 1
    print("  most common errors:")
    for e, c in errs.most_common(6):
        print(f"    {c:4d}  {e}")

In [ ]:
EVAL_ROWS = random.Random(EVAL_SEED).sample(TEST_ROWS, N_EVAL)
baseline_rate, baseline_records = evaluate(EVAL_ROWS, "BASELINE (adapter is a no-op)")
report(baseline_records, "baseline breakdown")

In [ ]:
# Read two failures before training. Five minutes here tells you which reward does the work.
shown = 0
for rec in baseline_records:
    if not rec["passed"] and shown < 2:
        shown += 1
        print("=" * 96)
        print("REQUIRED:", {k: rec["row"][k] for k in
              ("output_format", "require_wrapper_key", "require_code_block",
               "require_no_commentary", "top_level_count", "top_level_key")})
        print("ERRORS  :", rec["errors"][:3])
        print("-" * 96)
        print(rec["completion"][:600])

## 7. Reward functions — the actual teacher

Three rewards, all returning `[0, 1]`, taken from the official notebook:

| Reward | Question | Weight |
|---|---|---:|
| `json_format_reward` | Parseable, and raw/fenced exactly as requested? | 1.0 |
| `field_count_reward` | Right number of top-level fields per object? | 0.5 |
| `schema_validation_reward` | Shape, required keys, types, enums, bounds satisfied? | 2.0 |

**Why partial credit and not just pass/fail.** A single binary reward makes early groups
all-zero, all advantages zero, and the gradient vanishes. So `json_format_reward` pays
0.2 for the *wrong* form (better than unparseable), and `schema_validation_reward` decays
with the violation count, reserving its top 25% for fully-valid output.

**Why these guard against reward hacking.** Coverage gates partial credit: a bare `{}`
would otherwise harvest a permissive denominator, so required-key coverage multiplies it
to ~0. Under-length arrays lose credit proportionally, or fewer sloppy items would
out-score the requested count. We test both below.

> **Reward ≠ metric, deliberately.** These rewards score Nemotron rows against
> `schema_str` with `jsonschema`. The IFStruct validator in section 3 is a *stricter,
> different* checker (it also enforces fencing, commentary, wrapper keys, counts, and
> extraneous fields). Training optimises the proxy; section 9 measures the real thing.
> Keeping them separate is what makes the eval honest — but it also means a rising reward
> curve does **not** guarantee a rising IFStruct score. That is exactly the gap the
> post-training evaluation exists to detect.

In [ ]:
def completion_text(completion) -> str:
    """Normalise TRL conversational completions (and plain strings, so unit tests are easy)."""
    return completion if isinstance(completion, str) else completion[-1]["content"]


def extract_json(text: str):
    """Parse as raw JSON or as the first valid fenced block. Returns (obj, form)
    with form in {"direct", "fenced", None}."""
    text = text.strip()
    if text.startswith(("{", "[")) and text.endswith(("}", "]")):
        try:
            return json.loads(text), "direct"
        except json.JSONDecodeError:
            pass
    # Fences must sit on their own lines -- strict parsers reject a glued closing fence.
    for m in re.finditer(r"```(?:json)?\s*\n(.*?)\n\s*```", text, re.DOTALL | re.IGNORECASE):
        try:
            return json.loads(m.group(1).strip()), "fenced"
        except json.JSONDecodeError:
            continue
    return None, None

In [ ]:
def json_format_reward(completions, **kwargs) -> list[float]:
    scores = []
    for completion, fence_requested in zip(completions, kwargs["wants_fence"]):
        _, form = extract_json(completion_text(completion))
        requested = "fenced" if fence_requested else "direct"
        # 0.2 keeps the wrong form above unparseable output.
        scores.append(0.0 if form is None else 1.0 if form == requested else 0.2)
    return scores


def field_count_reward(completions, **kwargs) -> list[float]:
    scores = []
    for completion, expected in zip(completions, kwargs["schema_fields_count"]):
        expected = int(expected)          # the dataset stores this count as a string
        obj, _ = extract_json(completion_text(completion))
        # Arrays score the mean field count per object, keeping the per-object meaning.
        if isinstance(obj, list):
            dicts = [x for x in obj if isinstance(x, dict)]
            actual = sum(len(x) for x in dicts) / len(dicts) if dicts else None
        elif isinstance(obj, dict):
            actual = len(obj)
        else:
            actual = None

        if actual is None:
            scores.append(0.0)
        elif expected == 0:
            scores.append(1.0 if actual == 0 else 0.0)
        else:
            scores.append(max(0.0, 1.0 - abs(actual - expected) / expected))
    return scores

In [ ]:
def count_schema_checks(schema) -> int:
    """Count declared constraints recursively so nested violations weigh like top-level ones."""
    constraint_keywords = ("enum", "pattern", "minItems", "maxItems", "minimum", "maximum",
                           "minLength", "maxLength", "additionalProperties")
    if isinstance(schema, list):
        return sum(count_schema_checks(i) for i in schema)
    if not isinstance(schema, dict):
        return 0
    # Data-valued keywords (examples, default, const) may hold dicts that merely look like
    # schemas -- type-check before taking len() instead of trusting the meta-schema.
    properties, required = schema.get("properties"), schema.get("required")
    count = len(properties) if isinstance(properties, dict) else 0
    count += len(required) if isinstance(required, list) else 0
    count += sum(1 for k in constraint_keywords if k in schema)
    count += sum(count_schema_checks(v) for v in schema.values() if isinstance(v, (dict, list)))
    return count


def schema_validation_reward(completions, **kwargs) -> list[float]:
    scores = []
    for completion, schema_text in zip(completions, kwargs["schema_str"]):
        schema = json.loads(schema_text)
        obj, _ = extract_json(completion_text(completion))
        # Model output is untrusted: anything not an object or array scores 0.
        if not isinstance(obj, (dict, list)):
            scores.append(0.0)
            continue

        # Coverage gates partial credit so {}, [] or the wrong top-level shape score ~0.
        if (schema.get("type") == "array") != isinstance(obj, list):
            coverage = 0.0
        elif isinstance(obj, dict):
            required = schema.get("required", [])
            coverage = sum(k in obj for k in required) / len(required) if required else 1.0
        elif not obj:
            coverage = 0.0
        else:
            n_min = schema.get("minItems")
            coverage = min(len(obj), n_min) / n_min if n_min else 1.0
            items = schema.get("items")
            required = items.get("required", []) if isinstance(items, dict) else []
            if required:
                coverage *= sum(sum(k in i for k in required) for i in obj if isinstance(i, dict)) \
                            / (len(required) * len(obj))

        errors = list(Draft7Validator(schema).iter_errors(obj))
        num_checks = count_schema_checks(schema)
        if isinstance(obj, list) and schema.get("minItems"):
            num_checks += (schema["minItems"] - 1) * count_schema_checks(schema.get("items"))

        partial = coverage * max(0.0, 1.0 - len(errors) / max(1, num_checks))
        scores.append(0.75 * partial + (0.25 if not errors else 0.0))   # top 25% = fully valid
    return scores


REWARD_FUNCS = [json_format_reward, field_count_reward, schema_validation_reward]
REWARD_WEIGHTS = [1.0, 0.5, 2.0]

### Unit-test the rewards before spending GPU hours

The most expensive GRPO bug is a reward that always returns the same number, parses the
schema instead of the answer, or quietly pays out for `{}`. Catch it here.

In [ ]:
as_chat = lambda text: [{"role": "assistant", "content": text}]

toy_schema = json.dumps({
    "type": "object", "required": ["status", "count"],
    "properties": {"status": {"type": "string", "enum": ["ok", "error"]},
                   "count": {"type": "integer", "minimum": 1, "maximum": 3}},
    "additionalProperties": False,
})

samples = {
    "perfect_raw":    '{"status":"ok","count":2}',
    "perfect_fenced": '```json\n{"status":"ok","count":2}\n```',
    "wrong_type":     '{"status":"ok","count":"two"}',
    "empty":          '{}',
    "commentary":     'Here you go: {"status":"ok","count":2}',
    "backtick_spam":  "`" * 512,        # what a collapsed policy emits
}

rows = [{"case": name,
         "form_raw":    json_format_reward([as_chat(t)], wants_fence=[False])[0],
         "form_fenced": json_format_reward([as_chat(t)], wants_fence=[True])[0],
         "field_count": field_count_reward([as_chat(t)], schema_fields_count=[2])[0],
         "schema":      schema_validation_reward([as_chat(t)], schema_str=[toy_schema])[0]}
        for name, t in samples.items()]
display(pd.DataFrame(rows).set_index("case").round(3))

by = {r["case"]: r for r in rows}
assert by["perfect_raw"]["schema"] == 1.0,     "a valid object must score 1.0"
assert by["empty"]["schema"] == 0.0,           "'{}' must not harvest partial credit"
assert by["commentary"]["form_raw"] == 0.0,    "commentary must not parse as direct JSON"
assert by["backtick_spam"]["form_fenced"] == 0.0, "backtick spam must score 0"
assert by["perfect_raw"]["form_fenced"] == 0.2, "wrong form sits above unparseable, below correct"
print("reward unit tests passed")

In [ ]:
# Contract check against a real training row: correct length, all values in [0, 1].
row = train_ds[0]
probe = [as_chat("{}"), as_chat("not json at all")]
for fn in REWARD_FUNCS:
    scores = fn(probe,
                wants_fence=[row["wants_fence"]] * 2,
                schema_fields_count=[row["schema_fields_count"]] * 2,
                schema_str=[row["schema_str"]] * 2)
    print(f"  {fn.__name__:26s} {scores}")
    assert len(scores) == 2 and all(0.0 <= s <= 1.0 for s in scores)

## 8. GRPO configuration

These are the published values. Four are worth understanding rather than copying:

* **`temperature=1.1`** — hotter than you would ever serve. Identical completions inside a
  group mean identical rewards, zero advantages, and a wasted step.
* **`beta=0.01`** — a small KL penalty anchoring the policy to the reference model. Unlike
  a `beta=0.0` setup this *does* keep a reference model in memory; it is the price of the
  anchor, and it gives you a KL curve to diagnose with.
* **`mask_truncated_completions=False`** — truncated completions stay in the loss, so
  run-on output is penalised rather than ignored.
* **`steps_per_generation=2`** — with `per_device_train_batch_size=4` this generates
  `4 × 2 = 8` completions at a time, which is what makes 8 generations fit on a free T4.

### The batch arithmetic TRL actually enforces

```
generation_batch_size = per_device_bs × num_processes × steps_per_generation = 4 × 1 × 2 = 8
must be divisible by num_generations                                          = 8 ✓
```

A guard that checks anything else gives false confidence — `per_device_bs %
steps_per_generation`, for instance, happily passes configurations TRL then rejects.

In [ ]:
from trl import GRPOConfig

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-5,
    warmup_steps=max(1, int(0.1 * MAX_STEPS)),
    max_steps=MAX_STEPS,

    bf16=USE_BF16,
    fp16=not USE_BF16,                  # T4 = fp16; bf16 on Turing is a classic silent bug

    max_completion_length=MAX_COMPLETION_LENGTH,   # tight caps zero out whole groups
    mask_truncated_completions=False,
    num_generations=NUM_GENERATIONS,
    temperature=1.1,
    reward_weights=REWARD_WEIGHTS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # several prompt groups per update tame noisy advantages
    steps_per_generation=2,
    beta=0.01,

    logging_steps=1,
    save_steps=MAX_STEPS,
    report_to="none",                   # or "wandb" / "tensorboard"
)
training_args

In [ ]:
# Preflight. Each assertion guards a failure that is expensive to discover mid-run.
num_processes = 1
gen_batch = (training_args.per_device_train_batch_size * num_processes
             * training_args.steps_per_generation)

assert len(train_ds) >= 4, "not enough training rows survived filtering"
assert torch.cuda.is_available()
assert training_args.num_generations >= 2, "GRPO needs a group, not a single sample"
assert gen_batch % training_args.num_generations == 0, (
    f"generation_batch_size ({gen_batch}) must be divisible by "
    f"num_generations ({training_args.num_generations}) -- TRL rejects partial groups")
assert sum(p.numel() for p in model.parameters() if p.requires_grad) > 0, "nothing trainable"

print(f"generation batch {gen_batch} = {gen_batch // training_args.num_generations} prompt(s) "
      f"x {training_args.num_generations} generations")
print("train rows:", len(train_ds),
      "| trainable:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("GPU allocated/reserved GB:", round(torch.cuda.memory_allocated()/1e9, 2),
      round(torch.cuda.memory_reserved()/1e9, 2))

### What healthy training looks like

* `json_format` reward moves **first** — form is the easiest thing to learn;
* `schema_validation` follows as types and enums fall into line;
* KL lifts off zero after warmup without exploding;
* completion length **stabilises** rather than climbing;
* some zero-variance ("dead") groups are normal; a sustained value near 1 means there is
  no relative signal left and the run is spinning.

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,                 # already LoRA-wrapped
    reward_funcs=REWARD_FUNCS,
    args=training_args,
    train_dataset=train_ds,
)

torch.cuda.empty_cache()
started = time.time()
train_result = trainer.train()
elapsed_minutes = (time.time() - started) / 60
print(f"training finished in {elapsed_minutes:.1f} minutes")

In [ ]:
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
assert "reward" in logs, "no reward logs -- inspect trainer.state.log_history"
logs = logs[logs["reward"].notna()].set_index("step")
ROLL = min(10, max(1, len(logs)))       # each step covers few prompts; judge the trend

def raw_and_rolling(ax, series, label=None):
    (faint,) = ax.plot(series.index, series, alpha=0.25)
    ax.plot(series.index, series.rolling(ROLL, min_periods=1).mean(),
            color=faint.get_color(), label=label)
    return faint.get_color()

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)

ax = axes[0, 0]
for col in logs.columns:
    if col.startswith("rewards/") and col.endswith("/mean"):
        raw_and_rolling(ax, logs[col], col.removeprefix("rewards/").removesuffix("/mean"))
ax.set_title("Reward components"); ax.set_ylabel("mean reward"); ax.legend(fontsize=8)

ax = axes[0, 1]
color = raw_and_rolling(ax, logs["reward"], "combined (weighted)")
if "reward_std" in logs:
    m = logs["reward"].rolling(ROLL, min_periods=1).mean()
    s = logs["reward_std"].rolling(ROLL, min_periods=1).mean()
    ax.fill_between(logs.index, m - s, m + s, alpha=0.15, color=color, label="±1 group std")
ax.set_title("Combined reward"); ax.legend(fontsize=8)

ax = axes[1, 0]
if "kl" in logs:
    raw_and_rolling(ax, logs["kl"], "KL")
if "completions/mean_length" in logs:
    ax2 = ax.twinx(); raw_and_rolling(ax2, logs["completions/mean_length"], "length")
    ax2.set_ylabel("tokens"); ax2.legend(fontsize=8, loc="lower right")
ax.set_title("KL from reference + completion length"); ax.set_xlabel("step"); ax.legend(fontsize=8)

ax = axes[1, 1]
if "frac_reward_zero_std" in logs:
    raw_and_rolling(ax, logs["frac_reward_zero_std"], "dead groups (zero reward std)")
if "completions/clipped_ratio" in logs:
    raw_and_rolling(ax, logs["completions/clipped_ratio"], "truncated")
ax.set_title("Group health"); ax.set_ylim(-0.05, 1.05); ax.set_xlabel("step"); ax.legend(fontsize=8)

fig.suptitle(f"GRPO diagnostics — {RUN_NAME}"); fig.tight_layout(); plt.show()

## 9. Measure again — same rows, same decoding, official scorer

Nothing changes but the adapter.

In [ ]:
tuned_rate, tuned_records = evaluate(EVAL_ROWS, "AFTER GRPO (adapter active)")
report(tuned_records, "post-GRPO breakdown")

In [ ]:
def exact_sign_test(before, after):
    """Paired two-sided sign test over rows whose verdict changed (McNemar, exact)."""
    b = sum(1 for x, y in zip(before, after) if x and not y)      # regressions
    c = sum(1 for x, y in zip(before, after) if y and not x)      # fixes
    n = b + c
    if n == 0:
        return b, c, 1.0
    tail = sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n
    return b, c, min(1.0, 2 * tail)

before = [r["passed"] for r in baseline_records]
after = [r["passed"] for r in tuned_records]
regressed, fixed, p = exact_sign_test(before, after)
mean_ratio = lambda recs: sum(r["ratio"] for r in recs) / len(recs)

print(f"{'':22s}{'baseline':>10s}{'tuned':>10s}{'delta':>10s}")
print("-" * 52)
print(f"{'IFStruct pass rate':22s}{100*baseline_rate:9.1f}%{100*tuned_rate:9.1f}%"
      f"{100*(tuned_rate-baseline_rate):+9.1f}")
print(f"{'mean field-match':22s}{100*mean_ratio(baseline_records):9.1f}%"
      f"{100*mean_ratio(tuned_records):9.1f}%"
      f"{100*(mean_ratio(tuned_records)-mean_ratio(baseline_records)):+9.1f}")
print(f"\nfixed: {fixed}   regressed: {regressed}   exact paired sign test p = {p:.4f}")

se = (baseline_rate * (1 - baseline_rate) / len(EVAL_ROWS)) ** 0.5
print(f"\nOne-sample standard error at n={len(EVAL_ROWS)} is ~{100*se:.1f}pp, which is why the "
      f"paired test\nabove -- not the raw difference -- is the thing to read.")
print("Published reference (all 2,000 rows, llama.cpp stack): 22.6% -> 29.7%")
if SMOKE_TEST:
    print("\n*** SMOKE MODE: 5 steps. This delta is noise by construction. ***")

In [ ]:
# Where did it come from? Compare error histograms before and after.
def error_hist(records):
    c = Counter()
    for rec in records:
        for e in rec["errors"]:
            c[re.sub(r"['\"].*", "", e)[:52]] += 1
    return c

hb, ha = error_hist(baseline_records), error_hist(tuned_records)
keys = sorted(set(hb) | set(ha), key=lambda k: -(hb[k] + ha[k]))[:12]
print(f"{'error category':54s}{'before':>8s}{'after':>8s}{'delta':>8s}")
print("-" * 80)
for k in keys:
    print(f"{k:54s}{hb[k]:8d}{ha[k]:8d}{ha[k]-hb[k]:+8d}")

In [ ]:
# One row the adapter fixed, side by side.
for b, a in zip(baseline_records, tuned_records):
    if (not b["passed"]) and a["passed"]:
        print("PROMPT (truncated):\n", b["row"]["prompt"][:350], "\n")
        print("=" * 44, "BEFORE", "=" * 44, "\n", b["completion"][:450])
        print("errors:", b["errors"][:2])
        print("=" * 45, "AFTER", "=" * 45, "\n", a["completion"][:450])
        break
else:
    print("no newly-passing row in this sample (expected in smoke mode)")

## 10. Save the adapter, the merged model, and the run card

Save both forms: the **adapter** is a few MB and convenient for continued PEFT training;
the **merged** checkpoint is self-contained and ready for GGUF conversion in section 11.
Merging is irreversible for that in-memory object, so evaluate before you merge — which
is exactly the order used here.

The run card is not bureaucracy. Without the seed, versions, generation settings and
row-selection rule, a number you produce today cannot be compared with one you produce
next week.

In [ ]:
import peft, trl

ADAPTER_DIR, MERGED_DIR = f"{OUTPUT_DIR}-adapter", f"{OUTPUT_DIR}-merged"

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

run_card = {
    "base_model": MODEL_ID, "training_dataset": DATASET_ID,
    "held_out_eval": "LiquidAI/ifstruct-v1.0 (official ifstruct.validator)",
    "mode": RUN_NAME, "steps": MAX_STEPS, "num_generations": NUM_GENERATIONS,
    "learning_rate": training_args.learning_rate, "beta": training_args.beta,
    "temperature": training_args.temperature, "reward_weights": REWARD_WEIGHTS,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "targets": TARGET_MODULES},
    "seed": SEED, "dtype": str(DTYPE), "gpu": torch.cuda.get_device_name(0),
    "elapsed_minutes": round(elapsed_minutes, 2),
    "eval": {"n_rows": N_EVAL, "eval_seed": EVAL_SEED, "max_new_tokens": EVAL_MAX_NEW_TOKENS,
             "decoding": "greedy", "prompt_shape": "single user message, no system prompt",
             "baseline_pass_rate": baseline_rate, "tuned_pass_rate": tuned_rate,
             "fixed": fixed, "regressed": regressed, "sign_test_p": p},
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "trl": trl.__version__, "peft": peft.__version__},
}
Path(MERGED_DIR, "experiment.json").write_text(json.dumps(run_card, indent=2))
print(json.dumps(run_card, indent=2)[:1200])

## 11. The official 2,000-row evaluation

The subset score above is a **regression test**, not a publishable number: it uses fewer
rows and a smaller token budget than the official harness. To produce a figure comparable
to 22.6% / 29.7%, run the same stack the authors did — merge (done), convert to BF16
GGUF, serve with `llama-server`, and score with `ifstruct-eval`.

Do this in a local terminal; it is awkward inside free Colab.

In [ ]:
OFFICIAL_EVAL = "\n".join([
    "git clone --depth 1 https://github.com/ggml-org/llama.cpp",
    "pip install ./llama.cpp/gguf-py",
    f"python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \\",
    "  --outfile ./lfm25-350m-grpo-bf16.gguf --outtype bf16",
    "",
    "llama-server -m ./lfm25-350m-grpo-bf16.gguf \\",
    "  --alias lfm25-350m-grpo-structured-output -c 32768 -np 4 -ngl 99 \\",
    "  --host 127.0.0.1 --port 8081",
    "",
    "# in another terminal, from a clone of https://github.com/Liquid4All/ifstruct",
    "uv run ifstruct-eval \\",
    "  --model lfm25-350m-grpo-structured-output \\",
    "  --base-url http://localhost:8081/v1 --api-key dummy \\",
    "  --dataset data/test.jsonl --results-file results/lfm25-350m-grpo.json \\",
    "  --n-threads 4 --max-tokens 2048 -v",
])
print(OFFICIAL_EVAL)
print("\n# Report numerator/denominator (e.g. 594/2000), not just a percentage, and score")
print("# the BASE model through this identical stack for the 'before' number.")

## 12. Serving it

Three rules for structured output in production:

1. **Decode cold.** `do_sample=False`, or the temperature the model card recommends
   (`0.1`). The hot sampling above was for exploration during training only.
2. **Validate every response.** GRPO raises the pass rate; it does not make it 1.0. The
   validator costs ~1 ms. On failure, hand the model the *concrete* errors rather than
   retrying blind.
3. **Constrained decoding is complementary.** `outlines`, `xgrammar` and `llguidance` can
   force well-formed output, but they fight a policy that wants to write prose. GRPO
   reduces how often the grammar must intervene, so the two stack well — add the engine
   *after* the model has internalised the format, not instead of it.

In [ ]:
def structured_generate(user_prompt, spec, model_obj=None, max_retries=1,
                        max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Generate -> validate with the official checker -> repair once using the real errors."""
    model_obj = model_obj if model_obj is not None else merged_model
    messages = [{"role": "user", "content": user_prompt}]
    for attempt in range(max_retries + 1):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model_obj.device)
        with torch.inference_mode():
            out = model_obj.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                     pad_token_id=tokenizer.pad_token_id, use_cache=True)
        completion = tokenizer.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)

        verdict = score_one(spec, completion)
        if verdict.passed:
            return {"ok": True, "attempts": attempt + 1, "text": completion}
        if attempt == max_retries:
            return {"ok": False, "attempts": attempt + 1, "text": completion,
                    "errors": verdict.errors}
        messages += [{"role": "assistant", "content": completion},
                     {"role": "user", "content": "That response was rejected:\n- "
                      + "\n- ".join(verdict.errors[:5])
                      + "\nReturn only the corrected document."}]


probe = EVAL_ROWS[0]
result = structured_generate(probe["prompt"], probe)
print(f"passed={result['ok']}  attempts={result['attempts']}")
print(result["text"][:600])

## 13. Troubleshooting

| Symptom | Likely cause | Move |
|---|---|---|
| CUDA OOM during rollout | completion cap or group too large | lower `MAX_COMPLETION_LENGTH` first, then `NUM_GENERATIONS` 8→4; change one knob at a time |
| Reward flat, dead groups ≈ 1 | generations identical or all invalid | keep `temperature=1.1`; read raw completions; the partial-credit rewards exist for this |
| Reward flat from step 0 | LoRA matched nothing | the assertion in §5 catches this — `out_proj`, not `o_proj` |
| Reward rises, IFStruct doesn't | proxy/metric gap, or reward hacking | expected to a degree (§7); read completions, then check §9's error histogram |
| Repeated backticks/fences | reward hacking or truncation | keep `mask_truncated_completions=False`; tighten the parser |
| `{}` or empty arrays | permissive coverage | the coverage gate blocks this; re-run the §7 unit tests if you edited a reward |
| Valid JSON, wrong types | format learned before schema | normal ordering; keep the schema weight highest |
| Loss `nan` on a T4 | bf16 on Turing | `bf16=False, fp16=True` — detected automatically here |
| Output degrades as loss moves | LR too high / weak anchor | lower `5e-5`; raise `beta` |
| YAML doesn't improve | the recipe is JSON-only | expected — see the appendix |
| Full run ≠ 29.7% | seed, versions, serving stack | pin everything; score base and tuned through the *identical* stack |

**Rule:** log raw completions. Aggregate curves look healthy while a model exploits a
validator edge case.

## Appendix A — a YAML-capable extension (Run B, not the published reproduction)

IFStruct is half YAML, but this recipe trains JSON only — which is exactly why the
published YAML score barely moved (27.2% → 27.5%). Closing that gap needs training data
with YAML, fencing, wrapper-key and item-count metadata, and **you cannot take it from
IFStruct** without contaminating your test set.

The generator below builds that data: entity templates × a schema builder × five
presentation styles × the constraint axes, emitting rows in the exact column contract the
official validator consumes. Every row is then checked with `reference_answer()` — a row
whose reward is unreachable teaches nothing, and an all-zero group has no gradient at all.

Keep the experiments separable: **Run A** is the published JSON reproduction above;
**Run B** is this. A combined score is uninterpretable if the training distribution and
the evaluator moved at the same time.

In [ ]:
import json, random
from typing import Any

# ----------------------------------------------------------------------------- entities
def F(name, type_, **kw):
    d = {"name": name, "type": type_}
    d.update(kw)
    return d

ENTITY_SPECS = [
    {
        "key": "invoice", "label": "invoices", "singular": "invoice",
        "scenarios": ["a freelance design studio billing a retail client",
                      "a cloud hosting provider billing a startup",
                      "a logistics company billing a manufacturer"],
        "context": "Amounts should add up plausibly and identifiers should look like real invoice references.",
        "fields": [
            F("invoice_number", "string", desc="Invoice reference as printed on the document"),
            F("currency", "enum", values=["USD", "EUR", "GBP", "JPY"]),
            F("subtotal", "number", min=10, max=250000),
            F("tax_rate_pct", "number", min=0, max=27),
            F("total_due", "number", min=10, max=300000),
            F("days_until_due", "integer", min=0, max=180),
            F("paid", "boolean"),
            F("line_items", "object_array", item_min=2, item_max=4, desc="Individual billed lines", items=[
                F("description", "string", desc="What was billed"),
                F("quantity", "integer", min=1, max=500),
                F("unit_price", "number", min=0, max=50000),
            ]),
        ],
    },
    {
        "key": "gpu_review", "label": "GPU reviews", "singular": "GPU review",
        "scenarios": ["a mid-range card tested for 1440p gaming",
                      "a workstation card tested for CUDA rendering",
                      "a last-generation card tested for local LLM inference"],
        "context": "Benchmark numbers should be plausible for the class of card being described.",
        "fields": [
            F("model_name", "string", desc="Card name only, no vendor marketing suffix"),
            F("vram_gb", "integer", min=4, max=96),
            F("avg_fps_1440p", "number", min=10, max=400),
            F("power_draw_w", "integer", min=50, max=700),
            F("verdict", "enum", values=["buy", "wait", "skip"]),
            F("supports_ray_tracing", "boolean"),
            F("pros", "string_array", item_min=2, item_max=3, desc="Short positive points"),
        ],
    },
    {
        "key": "clinical_trial", "label": "clinical trial records", "singular": "clinical trial record",
        "scenarios": ["a phase II oncology study", "a phase III cardiology study",
                      "an early-phase vaccine immunogenicity study"],
        "context": "Enrollment figures and phase labels should be internally consistent.",
        "fields": [
            F("trial_title", "string", desc="Official study title"),
            F("phase", "enum", values=["I", "I/II", "II", "II/III", "III", "IV"]),
            F("enrolled_participants", "integer", min=8, max=15000),
            F("randomized", "boolean"),
            F("primary_endpoint", "string", desc="Primary outcome measure"),
            F("arms", "object_array", item_min=2, item_max=3, desc="Study arms", items=[
                F("arm_label", "string"),
                F("allocation", "enum", values=["treatment", "placebo", "active_comparator", "observation"]),
                F("participants", "integer", min=4, max=8000),
            ]),
        ],
    },
    {
        "key": "job_posting", "label": "job postings", "singular": "job posting",
        "scenarios": ["a remote backend role at a fintech scale-up",
                      "an on-site hardware role at a robotics company",
                      "a hybrid data role at a healthcare provider"],
        "context": "Compensation bands and seniority should be consistent with each other.",
        "fields": [
            F("job_title", "string", desc="Role title only, no company name"),
            F("seniority", "enum", values=["intern", "junior", "mid", "senior", "staff", "principal"]),
            F("salary_min_usd", "integer", min=20000, max=400000),
            F("salary_max_usd", "integer", min=25000, max=600000),
            F("remote_allowed", "boolean"),
            F("required_skills", "string_array", item_min=3, item_max=5, desc="Named technologies or skills"),
        ],
    },
    {
        "key": "conference_schedule", "label": "conference schedules", "singular": "conference schedule",
        "scenarios": ["a two-track systems conference", "a single-track design summit",
                      "an academic workshop day"],
        "context": "Session times and room labels should be plausible for a real event programme.",
        "fields": [
            F("track_name", "string", desc="Track or room programme name"),
            F("day_index", "integer", min=1, max=5),
            F("livestreamed", "boolean"),
            F("sessions", "object_array", item_min=2, item_max=4, desc="Talks in this track", items=[
                F("title", "string"),
                F("speaker", "string"),
                F("minutes", "integer", min=5, max=180),
                F("session_type", "enum", values=["keynote", "talk", "panel", "workshop", "lightning"]),
            ]),
        ],
    },
    {
        "key": "recipe", "label": "recipes", "singular": "recipe",
        "scenarios": ["a weeknight one-pan dinner", "a bakery-style breakfast pastry",
                      "a vegetarian batch-cook lunch"],
        "context": "Quantities and timings should be realistic for a home kitchen.",
        "fields": [
            F("title", "string", desc="Dish name only"),
            F("servings", "integer", min=1, max=12),
            F("total_minutes", "integer", min=5, max=480),
            F("difficulty", "enum", values=["easy", "medium", "hard"]),
            F("vegetarian", "boolean"),
            F("ingredients", "string_array", item_min=3, item_max=6, desc="Ingredient lines with quantities"),
            F("steps", "string_array", item_min=3, item_max=5, desc="Ordered preparation steps"),
        ],
    },
    # --- escaping-heavy entities (strings that must carry quotes / newlines / symbols)
    {
        "key": "bug_report_batch", "label": "bug reports", "singular": "bug report",
        "scenarios": ["a crash triage queue for a desktop app",
                      "a regression sweep after a dependency bump",
                      "an on-call queue for a payments service"],
        "context": ("Log excerpts should look like real captured output. Include quotation marks and literal "
                    "newline characters inside the excerpt text where natural."),
        "escaping": True,
        "fields": [
            F("summary", "string", desc="One-line issue summary"),
            F("severity", "enum", values=["blocker", "critical", "major", "minor", "trivial"]),
            F("reproducible", "boolean"),
            F("occurrences_last_7d", "integer", min=1, max=50000),
            F("log_excerpt", "string", desc='Raw captured log lines, including quotes and newlines'),
        ],
    },
    {
        "key": "dialogue_sample", "label": "dialogue samples", "singular": "dialogue sample",
        "scenarios": ["a support call transcript", "a two-hander scene from a stage play",
                      "an interview excerpt"],
        "context": ("Utterances should read naturally and contain apostrophes and quoted speech where the "
                    "exchange calls for it."),
        "escaping": True,
        "fields": [
            F("scene_label", "string", desc="Short label for the exchange"),
            F("register", "enum", values=["formal", "casual", "technical", "confrontational"]),
            F("turns", "object_array", item_min=3, item_max=5, desc="Speaking turns in order", items=[
                F("speaker", "string"),
                F("utterance", "string", desc='What the speaker says, including any quoted speech'),
                F("interrupted", "boolean"),
            ]),
        ],
    },
    {
        "key": "config_snippet_audit", "label": "config audits", "singular": "config audit",
        "scenarios": ["a hardening review of a reverse proxy", "a review of a CI pipeline definition",
                      "a review of a container runtime configuration"],
        "context": ("Config excerpts should be copied-out fragments with indentation, symbols and quoting "
                    "preserved exactly."),
        "escaping": True,
        "fields": [
            F("file_path", "string", desc="Path of the audited file"),
            F("risk_level", "enum", values=["none", "low", "medium", "high", "critical"]),
            F("auto_fixable", "boolean"),
            F("findings", "object_array", item_min=2, item_max=3, desc="Individual audit findings", items=[
                F("rule_id", "string"),
                F("excerpt", "string", desc="The offending configuration lines, verbatim"),
                F("line_number", "integer", min=1, max=20000),
            ]),
        ],
    },
    {
        "key": "api_endpoint_spec", "label": "API endpoint specs", "singular": "API endpoint spec",
        "scenarios": ["an internal billing service", "a public read-only catalogue API",
                      "an admin service behind mTLS"],
        "context": "Paths, methods and status codes should be coherent with each other.",
        "fields": [
            F("path", "string", desc="URL path template"),
            F("method", "enum", values=["GET", "POST", "PUT", "PATCH", "DELETE"]),
            F("auth_required", "boolean"),
            F("timeout_ms", "integer", min=50, max=120000),
            F("parameters", "object_array", item_min=2, item_max=4, desc="Accepted parameters", items=[
                F("name", "string"),
                F("location", "enum", values=["path", "query", "header", "body"]),
                F("required", "boolean"),
            ]),
        ],
    },
]

In [ ]:
# ----------------------------------------------------------------------------- schema
def _leaf_schema(f: dict) -> dict[str, Any]:
    t = f["type"]
    if t == "enum":
        s = {"type": "string", "enum": list(f["values"])}
    elif t in ("integer", "number"):
        s = {"type": t}
        if "min" in f: s["minimum"] = f["min"]
        if "max" in f: s["maximum"] = f["max"]
    elif t == "boolean":
        s = {"type": "boolean"}
    elif t == "string_array":
        s = {"type": "array", "items": {"type": "string"},
             "minItems": f["item_min"], "maxItems": f["item_max"]}
    elif t == "object_array":
        s = {"type": "array",
             "items": {"type": "object",
                       "properties": {c["name"]: _leaf_schema(c) for c in f["items"]},
                       "required": [c["name"] for c in f["items"]]},
             "minItems": f["item_min"], "maxItems": f["item_max"]}
    else:
        s = {"type": "string"}
    if f.get("desc") and t not in ("object_array",):
        s.setdefault("description", f["desc"])
    return s


def build_schema(fields: list[dict], count) -> dict[str, Any]:
    """IFStruct schemas are ALWAYS `{"type": "array", "items": {...}}` — the schema
    describes the *unwrapped* list of items, never the wrapper object."""
    lo, hi = (count, count) if isinstance(count, int) else (count[0], count[1])
    return {
        "type": "array",
        "items": {"type": "object",
                  "properties": {f["name"]: _leaf_schema(f) for f in fields},
                  "required": [f["name"] for f in fields]},
        "minItems": lo, "maxItems": hi,
    }

In [ ]:
# ----------------------------------------------------------------------------- prompt pieces
def _constraint_text(f: dict) -> str:
    t = f["type"]
    if t == "enum":
        return "one of: " + ", ".join(f["values"])
    if t in ("integer", "number"):
        if "min" in f and "max" in f: return f"between {f['min']} and {f['max']}"
        return ""
    if t in ("string_array", "object_array"):
        return f"{f['item_min']}-{f['item_max']} items"
    return ""


def _type_word(f: dict) -> str:
    return {"enum": "string", "string_array": "array of strings",
            "object_array": "array of objects"}.get(f["type"], f["type"])


def _count_phrase(count) -> str:
    return str(count) if isinstance(count, int) else f"{count[0]}-{count[1]}"


def _pseudo_value(f: dict, indent: str) -> str:
    t = f["type"]
    if t == "enum":
        return "|".join(f["values"])
    if t in ("integer", "number"):
        bits = []
        if "min" in f: bits.append(f"≥{f['min']}")
        if "max" in f: bits.append(f"≤{f['max']}")
        return f"<{t}>" + (f" ({', '.join(bits)})" if bits else "")
    if t == "boolean":
        return "<boolean>"
    if t == "string_array":
        return f"\n{indent}  - <string>  # {f['item_min']}-{f['item_max']} items\n{indent}  - ..."
    if t == "object_array":
        inner = []
        for i, c in enumerate(f["items"]):
            lead = f"{indent}  - " if i == 0 else f"{indent}    "
            inner.append(f"{lead}{c['name']}: {_pseudo_value(c, indent + '    ')}")
        return (f"  # {f['item_min']}-{f['item_max']} items\n" + "\n".join(inner) + f"\n{indent}  - ...")
    return "<string>" + (f"  # {f['desc']}" if f.get("desc") else "")

In [ ]:
# ----------------------------------------------------------------------------- 5 presentation styles
def style_md_table(fields, count, key, wrapper):
    rows = ["| path | constraints | notes | type |", "| --- | --- | --- | --- |"]
    for f in fields:
        rows.append(f"| `{f['name']}` | {_constraint_text(f)} | {f.get('desc','')} | {_type_word(f)} |")
        for c in f.get("items", []):
            rows.append(f"| `{f['name']}[].{c['name']}` | {_constraint_text(c)} | {c.get('desc','')} | {_type_word(c)} |")
    return "Requested fields:\n" + "\n".join(rows)


def style_bullets(fields, count, key, wrapper):
    out = ["Fields to include:"]
    for f in fields:
        con = _constraint_text(f)
        out.append(f"- `{f['name']}` ({_type_word(f)})" + (f" — {con}" if con else "")
                   + (f". {f['desc']}." if f.get("desc") else ""))
        for c in f.get("items", []):
            con2 = _constraint_text(c)
            out.append(f"  - `{f['name']}[].{c['name']}` ({_type_word(c)})" + (f" — {con2}" if con2 else ""))
    return "\n".join(out)


def style_raw_jsonschema(fields, count, key, wrapper):
    return ("Conform to this JSON Schema:\n\n```\n"
            + json.dumps(build_schema(fields, count), indent=2) + "\n```")


def style_pseudo_block(fields, count, key, wrapper):
    lines = ["Response structure:", "```"]
    if wrapper:
        lines.append(f"{key}:  # {_count_phrase(count)} items")
        ind, lead = "  ", "  - "
    else:
        lines.append(f"# {_count_phrase(count)} items")
        ind, lead = "", "- "
    for i, f in enumerate(fields):
        prefix = (lead if i == 0 else ind + "  ")
        lines.append(f"{prefix}{f['name']}: {_pseudo_value(f, ind + '  ')}")
    lines.append(f"{ind}- ...")
    lines.append("```")
    return "\n".join(lines)


def style_prose(fields, count, key, wrapper):
    parts = []
    for f in fields:
        con = _constraint_text(f)
        seg = f"{f['name']} as {_type_word(f)}"
        if con: seg += f" ({con})"
        if f.get("items"):
            seg += " where each entry has " + ", ".join(
                c["name"] + (f" [{_constraint_text(c)}]" if _constraint_text(c) else "") for c in f["items"])
        parts.append(seg)
    return "Each entry needs " + "; ".join(parts) + "."


STYLES = [style_md_table, style_bullets, style_raw_jsonschema, style_pseudo_block, style_prose]

In [ ]:
# ----------------------------------------------------------------------------- row assembly
FORMAT_LINES = {
    ("json", True):  ["Output valid JSON. Return an object with `{key}` as the key for the array.",
                      "Format your response as JSON. Use `{key}` as the top-level key wrapping the array.",
                      "Your response should be JSON, wrapped in an object under the key `{key}`."],
    ("json", False): ["Output valid JSON. Return a bare list at the top level, not wrapped in an object.",
                      "Use JSON format for your response. Return a bare list at the top level, not wrapped in an object.",
                      "Provide the output in JSON as a top-level array (no wrapper object)."],
    ("yaml", True):  ["Format your response as YAML. Return an object with `{key}` as the key for the array.",
                      "Respond in YAML with `{key}` as the top-level key for the array.",
                      "Use YAML. The array should sit under the top-level key `{key}`."],
    ("yaml", False): ["Format your response as YAML. Return a bare list at the top level, not wrapped in an object.",
                      "Respond in block-style YAML as a bare top-level list, not wrapped in an object.",
                      "Use YAML for your response. Top level must be a plain list, no wrapper key."],
}
FENCE_LINES = {
    True: {"json": ["Put the JSON in a ```json fenced code block.", "Enclose the JSON in a ```json code block.",
                    "Wrap your response in a code block.", "Use a code block for your response."],
           "yaml": ["Put the YAML in a ```yaml fenced code block.", "Enclose the YAML in a ```yaml code block.",
                    "Format your response inside a ```yaml code block.", "Wrap your YAML in a ```yaml code block."]},
    False: {"json": ["Output plain JSON without wrapping in a code block.",
                     "No code block needed - output the JSON directly.",
                     "Output the raw JSON directly without code block fencing."],
            "yaml": ["Output plain YAML without wrapping in a code block.",
                     "No code block needed - output the YAML directly.",
                     "Output the raw YAML directly without code block fencing."]},
}
NO_COMMENT_LINES = {
    "json": ["Respond with just the JSON, no explanations.", "Just the JSON, no commentary or preamble.",
             "Return only the JSON document - no preface, no trailing notes."],
    "yaml": ["Respond with just the YAML, no explanations.", "Just the YAML, no commentary or preamble.",
             "Return only the YAML document - no preface, no trailing notes."],
}


def make_example(rng: random.Random, seed: int) -> dict[str, Any]:
    spec = rng.choice(ENTITY_SPECS)
    fmt = rng.choice(["json", "yaml"])
    wrapper = rng.random() < 0.5
    fence = rng.random() < 0.63          # ifstruct test set is ~63% fenced
    no_comment = rng.random() < 0.5
    count = rng.choice([1, 2, 3, 4, [1, 2], [1, 3], [2, 3], [2, 4], [3, 4]])

    # Keep a subset of fields so prompts vary in width (always >= 4).
    fields = list(spec["fields"])
    if len(fields) > 4 and rng.random() < 0.45:
        keep = rng.randint(4, len(fields))
        fields = fields[:keep]

    key = spec["key"] if rng.random() < 0.5 else spec["key"] + "s"
    schema = build_schema(fields, count)
    scenario = rng.choice(spec["scenarios"])

    head = (f"Generate {_count_phrase(count)} {spec['label'] if not isinstance(count, int) or count > 1 else spec['singular']} "
            f"for {scenario}.")
    body = [head, f"Assign field values consistent with {scenario}.", spec["context"]]

    fmt_line = rng.choice(FORMAT_LINES[(fmt, wrapper)]).format(key=key)
    lines = ["\n".join(body), "", fmt_line, rng.choice(FENCE_LINES[fence][fmt]), "",
             rng.choice(STYLES)(fields, count, key, wrapper)]
    if no_comment:
        lines += ["", rng.choice(NO_COMMENT_LINES[fmt])]
    if rng.random() < 0.35:
        lines += ["", "Match the requested schema exactly and do not add extra keys."]

    return {
        "seed": seed,
        "entity_type": f"train__{spec['key']}",
        "prompt": "\n".join(lines).strip(),
        "json_schema": schema,
        "top_level_count": count,
        "top_level_key": key,
        "require_wrapper_key": wrapper,
        "require_code_block": fence,
        "require_no_commentary": no_comment,
        "output_format": fmt,
    }


def build_trainset(n: int, seed: int = 0) -> list[dict[str, Any]]:
    rng = random.Random(seed)
    return [make_example(rng, seed=i) for i in range(n)]

In [ ]:
# Build it, then PROVE every row is satisfiable under the official validator.
yaml_train_rows = build_trainset(500, seed=0)
unsat = [r for r in yaml_train_rows if not score_one(r, reference_answer(r)).passed]
print(f"solvability: {len(yaml_train_rows)-len(unsat)}/{len(yaml_train_rows)} rows satisfiable")
assert not unsat, "unsatisfiable rows -- fix the generator before training on it"

overlap = {r["prompt"] for r in yaml_train_rows} & {r["prompt"] for r in TEST_ROWS}
print("prompt overlap with the IFStruct test set:", len(overlap))
assert not overlap, "CONTAMINATION: generated prompts collide with the test set"

for col in ["output_format", "require_wrapper_key", "require_code_block", "require_no_commentary"]:
    print(f"  {col:24s} {dict(Counter(r[col] for r in yaml_train_rows))}")
print("\n" + "=" * 96 + "\n" + yaml_train_rows[3]["prompt"][:900])

### Wiring Run B up

To train on these rows you need a reward that speaks the IFStruct contract rather than
Nemotron's `schema_str`. The natural choice is the official validator itself — the same
one section 3 scores with — decomposed into partial credit so early groups are not all
zero:

```python
from datasets import Dataset

def to_hf(rows):                       # Arrow cannot hold `int | list` or varying
    return Dataset.from_list([{        # schemas in one column -> store JSON strings
        "prompt": [{"role": "user", "content": r["prompt"]}],
        "json_schema":     json.dumps(r["json_schema"]),
        "top_level_count": json.dumps(r["top_level_count"]),
        "output_format":   r["output_format"],
        "top_level_key":   r["top_level_key"],
        "require_wrapper_key":   bool(r["require_wrapper_key"]),
        "require_code_block":    bool(r["require_code_block"]),
        "require_no_commentary": bool(r["require_no_commentary"]),
    } for r in rows])

def reward_ifstruct_pass(completions, **kw):
    rows = [{k: kw[k][i] for k in kw} for i in range(len(completions))]
    return [float(score_one(r, completion_text(c)).passed) for c, r in zip(completions, rows)]

def reward_schema_fields(completions, **kw):    # dense: fraction of leaf fields valid
    rows = [{k: kw[k][i] for k in kw} for i in range(len(completions))]
    return [score_one(r, completion_text(c)).details.get("schema_match_ratio", 0.0)
            for c, r in zip(completions, rows)]
```

Note the `json.dumps` on those two columns — Arrow raises `ArrowInvalid: cannot mix list
and non-list, non-null values` otherwise, because `top_level_count` is an `int` on some
rows and a `[min, max]` list on others. Cache the validator call if you add more than two
rewards, or you will pay for the same parse repeatedly.

## Appendix B — reproducibility checklist

- [ ] GPU model and memory recorded
- [ ] package versions pinned and captured in the run card
- [ ] base model revision recorded
- [ ] dataset revision and row-selection rule recorded
- [ ] IFStruct never used for training or for tuning reward thresholds
- [ ] seed, reward code, weights, LoRA targets, generation settings saved
- [ ] raw completions sampled and eyeballed during training
- [ ] adapter **and** merged model saved
- [ ] base and tuned scored through the *identical* stack
- [ ] full 2,000-row numerator/denominator reported, not only a percentage

### Where to go next

1. **More data before more steps** — 500 rows is the binding constraint, not 100 steps.
2. **250–500 steps** at `num_generations=8` for a lower-variance advantage estimate.
3. **Curriculum**: shallow schemas first, deep nesting later.
4. **Then** tune `r=32`, the LR, and `beta`.

Re-run the *whole* evaluation at each change, and read the paired test rather than the
raw difference.

## References

* [*Fine-tuning a 350M Model for Better Structured Outputs in 100 GRPO Steps*](https://huggingface.co/blog/grpo-with-trl-ifstruct) — Monigatti, Burtenshaw & Paniego, 3 Sep 2026
* [Official notebook](https://github.com/Liquid4All/cookbook/blob/main/finetuning/notebooks/grpo_with_trl_ifstruct.ipynb) — `Liquid4All/cookbook`
* [*IFStruct: Measuring structured-output compliance*](https://www.liquid.ai/blog/ifstruct-v1.0) · [`Liquid4All/ifstruct`](https://github.com/Liquid4All/ifstruct) · [`LiquidAI/ifstruct-v1.0`](https://huggingface.co/datasets/LiquidAI/ifstruct-v1.0)
* [`LiquidAI/LFM2.5-350M`](https://huggingface.co/LiquidAI/LFM2.5-350M) · [`nvidia/Nemotron-RL-instruction_following-structured_outputs`](https://huggingface.co/datasets/nvidia/Nemotron-RL-instruction_following-structured_outputs)
* [TRL `GRPOTrainer`](https://huggingface.co/docs/trl/en/grpo_trainer) · [DeepSeekMath (GRPO)](https://arxiv.org/abs/2402.03300)